# Notebook for validating utilities located in the utils folder
This notebook inspects and validates the `utils` package within the project.

## 1. Set Up Environment
Configure import paths and ensure required dependencies are available.

In [1]:
import os
import sys
from pathlib import Path
import pkgutil
import importlib
import inspect
import json
import subprocess
PROJECT_ROOT = Path('d:/College/Major Project').resolve()
if PROJECT_ROOT.as_posix() not in map(lambda p: Path(p).as_posix(), sys.path):
    sys.path.insert(0, str(PROJECT_ROOT))
UTILS_PATH = PROJECT_ROOT / 'utils'
print(f"Project root: {PROJECT_ROOT}")
print(f"Utils package path: {UTILS_PATH}")
assert UTILS_PATH.exists(), "Utils directory not found"
def ensure_package(pkg_name: str):
    try:
        __import__(pkg_name)
        print(f"✅ {pkg_name} available")
    except ImportError:
        print(f"⚠️ {pkg_name} missing; attempting installation...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg_name], check=False)
        try:
            __import__(pkg_name)
            print(f"✅ {pkg_name} installed")
        except ImportError:
            print(f"❌ {pkg_name} still unavailable")
for dependency in ['torch', 'timm', 'fastapi']:
    ensure_package(dependency)

Project root: D:\College\Major Project
Utils package path: D:\College\Major Project\utils
✅ torch available
⚠️ timm missing; attempting installation...


c:\Users\anush\anaconda3\envs\crowdenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ timm installed
✅ fastapi available


## 2. Discover Utils Package Structure
Enumerate modules and top-level members within `utils`.

In [2]:
def iter_modules(base_package: str):
    package = importlib.import_module(base_package)
    base_path = Path(package.__file__).parent
    print(f"Scanning {base_package} at {base_path}")
    for module_info in pkgutil.walk_packages(package.__path__, prefix=f"{base_package}."):
        yield module_info
module_summary = []
for module_info in iter_modules('utils'):
    try:
        module = importlib.import_module(module_info.name)
        public_members = [name for name, obj in inspect.getmembers(module) if not name.startswith('_')]
        undocumented = [name for name in public_members if not inspect.getdoc(getattr(module, name))]
        module_summary.append({'module': module_info.name, 'members': public_members, 'undocumented': undocumented})
    except Exception as exc:
        module_summary.append({'module': module_info.name, 'error': str(exc)})
module_summary

Scanning utils at D:\College\Major Project\utils


[{'module': 'utils.checkup',
  'members': ['analyze_checkpoint_structure', 're', 'torch'],
  'undocumented': ['analyze_checkpoint_structure']},
 {'module': 'utils.postprocess',
  'members': ['F',
   'Optional',
   'Tuple',
   'analyze_density_map',
   'balanced_count_calibration',
   'cv2',
   'find_local_maxima',
   'get_count_from_density',
   'get_count_with_classification',
   'logger',
   'logging',
   'np',
   'postprocess_density_map',
   'torch',
   'visualize_density_map'],
  'undocumented': []},
 {'module': 'utils.preprocess',
  'members': ['Any',
   'Callable',
   'CustomTransforms',
   'F',
   'Image',
   'Iterable',
   'annotations',
   'np',
   'preprocess_frame',
   'torch',
   'transforms'],
  'undocumented': ['CustomTransforms', 'Image', 'annotations', 'transforms']},
 {'module': 'utils.visualize',
  'members': ['Optional',
   'Tuple',
   'base64',
   'create_density_statistics_overlay',
   'cv2',
   'generate_heatmap_overlay',
   'generate_pure_heatmap',
   'logger',


In [3]:
from pprint import pprint
pprint(module_summary)

[{'members': ['analyze_checkpoint_structure', 're', 'torch'],
  'module': 'utils.checkup',
  'undocumented': ['analyze_checkpoint_structure']},
 {'members': ['F',
              'Optional',
              'Tuple',
              'analyze_density_map',
              'balanced_count_calibration',
              'cv2',
              'find_local_maxima',
              'get_count_from_density',
              'get_count_with_classification',
              'logger',
              'logging',
              'np',
              'postprocess_density_map',
              'torch',
              'visualize_density_map'],
  'module': 'utils.postprocess',
  'undocumented': []},
 {'members': ['Any',
              'Callable',
              'CustomTransforms',
              'F',
              'Image',
              'Iterable',
              'annotations',
              'np',
              'preprocess_frame',
              'torch',
              'transforms'],
  'module': 'utils.preprocess',
  'undocumented': [

## 3. Load Utility Functions
Import key utilities and inspect their signatures and docstrings.

In [5]:
from typing import _SpecialGenericAlias, _SpecialForm
def is_callable_member(member):
    if callable(member):
        return True
    # Handle numpy ufuncs etc
    return hasattr(member, '__call__') and not isinstance(member, (_SpecialGenericAlias, _SpecialForm))
def describe_callable(obj, name):
    try:
        sig = str(inspect.signature(obj))
    except (TypeError, ValueError):
        sig = 'Signature unavailable'
    try:
        doc = inspect.getdoc(obj) or 'No docstring'
    except Exception:
        doc = 'Docstring unavailable'
    return {'name': name, 'signature': sig, 'doc': doc}
targets = ['utils.preprocess','utils.postprocess','utils.checkup','utils.visualize']
descriptions = {}
for target in targets:
    try:
        module = importlib.import_module(target)
        callables = {}
        for name, member in inspect.getmembers(module):
            if name.startswith('_'):
                continue
            if isinstance(member, (_SpecialGenericAlias, _SpecialForm)):
                continue
            if callable(member):
                callables[name] = describe_callable(member, name)
        descriptions[target] = callables or {'info': 'No callable members detected'}
    except Exception as exc:
        descriptions[target] = {'error': str(exc)}
descriptions

{'utils.preprocess': {'CustomTransforms': {'name': 'CustomTransforms',
   'signature': '()',
   'doc': 'No docstring'},
  'preprocess_frame': {'name': 'preprocess_frame',
   'signature': "(image: 'Any', max_long_edge: 'int' = 1536) -> 'torch.Tensor'",
   'doc': "Preprocess image for crowd counting model.\n\nArgs:\n    image: Input image in PIL, numpy (HWC, BGR), or grayscale formats.\n    max_long_edge: Maximum size for the image's longest edge. If the\n        provided image exceeds this, it will be downscaled while preserving\n        aspect ratio to accelerate inference.\n\nReturns:\n    torch.Tensor: Normalized image tensor in BCHW format (batch dimension\n    included)."}},
 'utils.postprocess': {'analyze_density_map': {'name': 'analyze_density_map',
   'signature': '(density_map: numpy.ndarray) -> dict',
   'doc': 'Analyze density map characteristics to inform calibration\n\nArgs:\n    density_map: 2D density map from model\n\nReturns:\n    Dictionary with analysis results'},
  '

In [ ]:
pprint(descriptions)

## 4. Run Targeted Unit Checks
Execute focused tests for the `utils` package using pytest (if available).

In [6]:
import shutil
PYTEST = shutil.which('pytest')
if PYTEST is None:
    print('⚠️ pytest not found; skipping unit checks.')
else:
    completed = subprocess.run([PYTEST, 'utils', '-q'], cwd=PROJECT_ROOT)
    print(f"pytest exit code: {completed.returncode}")

pytest exit code: 5


## 5. Execute Sample Scenarios
Run representative utility calls to validate behavior.

In [9]:
from PIL import Image
import numpy as np
import torch
checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'jhu_5.pth'
ckpt = torch.load(checkpoint_path, map_location='cpu')
if isinstance(ckpt, dict):
    keys = list(ckpt.keys())
    print(f'Checkpoint type: dict with keys {keys[:10]} (total {len(keys)})')
else:
    print(f'Checkpoint type: {type(ckpt)}')
from utils.preprocess import preprocess_frame
sample_image = Image.fromarray(np.random.randint(0,255,(720,1280,3),dtype=np.uint8))
tensor = preprocess_frame(sample_image)
print('Tensor shape:', tensor.shape, 'dtype:', tensor.dtype)
from utils.postprocess import analyze_density_map
dummy_map = tensor.mean(dim=1, keepdim=True)
analysis = analyze_density_map(dummy_map.squeeze().numpy())
analysis

Checkpoint type: dict with keys ['vmamba.patch_embed.0.weight', 'vmamba.patch_embed.0.bias', 'vmamba.patch_embed.2.weight', 'vmamba.patch_embed.2.bias', 'vmamba.patch_embed.5.weight', 'vmamba.patch_embed.5.bias', 'vmamba.patch_embed.7.weight', 'vmamba.patch_embed.7.bias', 'vmamba.layers.0.blocks.0.norm.weight', 'vmamba.layers.0.blocks.0.norm.bias'] (total 423)
Tensor shape: torch.Size([1, 3, 720, 1280]) dtype: torch.float32


{'total_sum': 200198.88,
 'max_value': 2.393062,
 'mean_nonzero': 0.69033265,
 'low_coverage': 0.6087141927083334,
 'med_coverage': 0.6041948784722222,
 'high_coverage': 0.5845648871527778,
 'num_peaks': 36987}

## 6. Automate Regression Snapshot
Persist current sample outputs for future regression comparisons.

In [11]:
snapshot_path = PROJECT_ROOT / 'utils' / 'checkpoint_snapshot.json'
def to_python(obj):
    if isinstance(obj, dict):
        return {k: to_python(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_python(v) for v in obj]
    if hasattr(obj, 'item'):
        try:
            return obj.item()
        except Exception:
            return str(obj)
    return obj
snapshot = {'analysis': to_python(analysis), 'tensor_shape': list(map(int, tensor.shape)), 'tensor_dtype': str(tensor.dtype)}
snapshot_path.write_text(json.dumps(snapshot, indent=2))
print(f'Snapshot saved to {snapshot_path}')

Snapshot saved to D:\College\Major Project\utils\checkpoint_snapshot.json


In [12]:
any(key.startswith('cls_head') for key in keys)
any(key.startswith('reg_head') for key in keys)
keys[-10:]

['reg_head.count.decoder.5.running_mean',
 'reg_head.count.decoder.5.running_var',
 'reg_head.count.decoder.5.num_batches_tracked',
 'reg_head.count.decoder.8.weight',
 'reg_head.count.decoder.9.weight',
 'reg_head.count.decoder.9.bias',
 'reg_head.count.decoder.9.running_mean',
 'reg_head.count.decoder.9.running_var',
 'reg_head.count.decoder.9.num_batches_tracked',
 'reg_head.count.decoder.11.weight']